Step 1) Load and build your “evidence corpus”

Each evidence item needs:

doc_id

risk_group (low / average / high)

doc_type (profile or snippet)

text

✅ For you: profile docs are 3; snippets are 150.

In [ ]:
import json, re
import pandas as pd

PROFILE_MD = "cluster_profile_docs.md"
SNIPPETS_JSONL = "snippet_bank_reports.jsonl"

def load_profile_docs(md_path):
    # Extract text blocks inside ```text ... ```
    with open(md_path, "r", encoding="utf-8") as f:
        md = f.read()

    blocks = re.findall(r"```text\n(.*?)```", md, flags=re.DOTALL)
    docs = []
    for b in blocks:
        b = b.strip()
        # Expect first line like: "CLUSTER PROFILE — low_risk (n=...)"
        m = re.search(r"CLUSTER PROFILE —\s*(\w+)", b)
        risk = m.group(1) if m else "unknown"
        docs.append({
            "doc_id": f"profile_{risk}",
            "risk_group": risk,
            "doc_type": "profile",
            "text": re.sub(r"\s+", " ", b).strip()
        })
    return docs

def load_snippet_docs(jsonl_path):
    docs = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            docs.append({
                "doc_id": row["snippet_id"],
                "risk_group": row["risk_group"],
                "doc_type": "snippet",
                "text": re.sub(r"\s+", " ", row["snippet_text"]).strip()
            })
    return docs

docs = load_profile_docs(PROFILE_MD) + load_snippet_docs(SNIPPETS_JSONL)
print("Total docs:", len(docs))  # should be 153


Total docs: 152


Step 2) Build TF-IDF index (easy, reproducible)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = [d["text"] for d in docs]

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),     # helps capture phrases
    min_df=1
)
X_tfidf = tfidf.fit_transform(texts)  # sparse (N_docs × vocab)


Build Embedding index

In [ ]:
pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, common
E = model.encode(texts, normalize_embeddings=True)  # (N_docs × dim)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Step 4) Define normalization + weighted hybrid score

You must normalize because TF-IDF and embeddings can have different score ranges.

Simple and common normalization per query:

𝑠
^
=
𝑠
−
min
⁡
(
𝑠
)
max
⁡
(
𝑠
)
−
min
⁡
(
𝑠
)
+
𝜖
s
^
=
max(s)−min(s)+ϵ
s−min(s)
	​


In [ ]:
import numpy as np

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min()) / (a.max() - a.min() + eps)

def hybrid_retrieve(query, risk_group=None, top_k=8, alpha=0.5):
    # 1) Candidate filtering (optional but recommended)
    idxs = list(range(len(docs)))
    if risk_group is not None:
        idxs = [i for i in idxs if docs[i]["risk_group"] == risk_group]

    # 2) TF-IDF similarity
    q_t = tfidf.transform([query])
    s_tfidf = cosine_similarity(q_t, X_tfidf[idxs]).ravel()

    # 3) Embedding similarity
    q_e = model.encode([query], normalize_embeddings=True)
    s_emb = (E[idxs] @ q_e[0])  # cosine since normalized

    # 4) Normalize and combine
    s_t = minmax_norm(s_tfidf)
    s_e = minmax_norm(s_emb)
    s_h = alpha * s_t + (1 - alpha) * s_e

    # 5) Rank
    order = np.argsort(-s_h)[:top_k]
    results = []
    for j in order:
        i = idxs[j]
        results.append({
            "doc_id": docs[i]["doc_id"],
            "risk_group": docs[i]["risk_group"],
            "doc_type": docs[i]["doc_type"],
            "hybrid": float(s_h[j]),
            "tfidf": float(s_t[j]),
            "emb": float(s_e[j]),
            "text": docs[i]["text"]
        })
    return results


Step 5) Use a good query (yours is fine)

Your query is good. I’d slightly expand it to match the style of your reports:

In [ ]:
query = ("hypertension blood pressure systolic diastolic BMI obesity "
         "cholesterol glucose diabetes risk factors cardiovascular")

Step 6) Practical “RAG-ready” retrieval rule

When summarizing cluster C, retrieve:

Always include the cluster profile doc: profile_C

Plus top-k snippets from that cluster via hybrid retrieval

That ensures the LLM always sees numbers + examples.

How to choose α (alpha) and k
Alpha (weight)

Start with α = 0.5

If your reports use consistent keywords (“BMI”, “cholesterol”), TF-IDF is strong → try α = 0.6–0.7

If wording varies a lot → embeddings matter more → try α = 0.3–0.5

k (how many snippets)

With 150 patients total, per cluster ~50.

For an LLM prompt, start with k = 6–10 snippets + the profile doc.

hybrid retrieval → prompt text

In [ ]:
def build_evidence_block(cluster_name, top_k=8, alpha=0.5):
    query = ("hypertension blood pressure systolic diastolic BMI obesity "
             "cholesterol glucose diabetes risk factors cardiovascular")

    # 1) retrieve top-k snippets from that cluster using your hybrid retriever
    snips = hybrid_retrieve(query, risk_group=cluster_name, top_k=top_k, alpha=alpha)

    # 2) get the profile doc for that cluster from your docs list
    prof = next(d for d in docs if d["doc_type"]=="profile" and d["risk_group"]==cluster_name)

    # 3) build evidence block with compact IDs
    lines = []
    lines.append("EVIDENCE (You MUST cite these IDs; do not invent new ones)\n")
    lines.append(f"[C] Cluster profile ({cluster_name}):\n{prof['text']}\n")

    for i, r in enumerate(snips, start=1):
        lines.append(f"[S{i}] Patient snippet:\n{r['text']}\n")

    return "\n".join(lines)

def build_user_prompt(cluster_name, evidence_block):
    return f"""Task: Write a cluster-level summary for cluster: {cluster_name}.

Output format (exact headings):
Cluster Overview
- (2–4 bullet sentences with citations)

Key Indicators (BP / BMI / Cholesterol / Glucose / Kidney)
- Blood Pressure: ...
- BMI: ...
- Cholesterol: ...
- Glucose: ...
- Kidney: ... (if not present → "insufficient evidence")

Main Risk Factors
- (3–6 bullet sentences with citations)

Suggested follow-ups (non-diagnostic, generic, grounded)
- (3–5 bullets). Each bullet must be justified by observed indicators (cite [C] and/or snippets).
- No medication advice. No diagnoses.

EVIDENCE:
{evidence_block}
"""


In [ ]:
# =========================
# Alpha sweep: test which alpha works best
# alpha=0.0  -> pure EMBEDDING retrieval
# alpha=1.0  -> pure TF-IDF retrieval
# (because hybrid score uses: s_h = alpha*s_tfidf + (1-alpha)*s_emb)
# =========================

import time
import pandas as pd
import numpy as np
import re, json, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# ---- Settings (keep fixed for fairness) ----
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
CLUSTERS = ["low_risk", "average_risk", "high_risk"]
TOP_K = 8
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0  # important: reduce randomness when comparing alphas

client = OpenAI()
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY before running this cell."

# ---- Step-7 style evaluator (minimal, no reference required) ----
PROFILE_TABLE = "cluster_profile_table.csv"
profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese",
              "pct_Glucose_prediabetes", "pct_Glucose_diabetes",
              "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b",
                     r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bchronic kidney disease\b", r"\bckd\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdosage\b", r"\bstatin\b", r"\binsulin\b", r"\bmetformin\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def extract_claims(summary_text: str, min_chars=20, max_claims=12):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    return claims[:max_claims]

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E_ = X[:len(e_texts)]
    C_ = X[len(e_texts):]
    sims = cosine_similarity(C_, E_)  # (n_claims x n_evidence)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i, j])
        results.append({
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds))

def count_patterns(text, patterns):
    t = text.lower()
    return sum(len(re.findall(pat, t)) for pat in patterns)

def evaluate_rag_summary(cluster_name, rag_summary, evidence_block):
    evidence_items = parse_evidence_block(evidence_block)
    claims = extract_claims(rag_summary, max_claims=12)
    checks = support_check_claims(claims, evidence_items, sim_threshold=0.22)
    supported = sum(1 for r in checks if r["supported"])
    total = len(checks) if len(checks)>0 else 1
    supported_rate = 100.0 * supported / total
    unsupported_rate = 100.0 - supported_rate

    must_inds = pick_must_mentions(cluster_name)
    comp = 100.0 * completeness_score(rag_summary, must_inds)

    abs_ct = count_patterns(rag_summary, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag_summary, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag_summary, MEDICATION_PATTERNS)

    return {
        "cluster": cluster_name,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "completeness_%": comp,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct,
    }

# ---- Generation helper ----
def generate_one_cluster(cluster_name, alpha):
    evidence_block = build_evidence_block(cluster_name, top_k=TOP_K, alpha=alpha)
    user_text = build_user_prompt(cluster_name, evidence_block)
    resp = client.responses.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        input=[
            {"role":"system", "content": system_text},
            {"role":"user", "content": user_text},
        ],
    )
    return resp.output_text, evidence_block

# ---- Run sweep ----
rows = []
outputs = []  # store full texts for qualitative checks
for a in ALPHAS:
    for c in CLUSTERS:
        summary_text, ev_block = generate_one_cluster(c, a)
        m = evaluate_rag_summary(c, summary_text, ev_block)
        m["alpha"] = a
        rows.append(m)
        outputs.append({
            "alpha": a,
            "cluster": c,
            "summary": summary_text,
            "evidence_block": ev_block
        })
        time.sleep(0.2)  # light pacing (optional)

df_alpha = pd.DataFrame(rows)
df_alpha.to_csv("alpha_sweep_results.csv", index=False)

# Macro averages per alpha
macro = df_alpha.groupby("alpha")[[
    "supported_claim_rate_%","unsupported_claim_rate_%","completeness_%",
    "absolute_claim_flags","diagnosis_like_flags","medication_flags"
]].mean().reset_index()

macro.to_csv("alpha_sweep_macro.csv", index=False)

# Rank alphas: highest supported-claim %, then highest completeness, then lowest absolute flags
macro_sorted = macro.sort_values(
    by=["supported_claim_rate_%","completeness_%","absolute_claim_flags"],
    ascending=[False, False, True]
)

print("Saved: alpha_sweep_results.csv and alpha_sweep_macro.csv")
display(macro_sorted)

# Also save the generated summaries + evidence for your qualitative section
with open("alpha_sweep_outputs.json", "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2)
print("Saved: alpha_sweep_outputs.json")


OpenAI API :

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

# NOTE: Set your API key in the environment (Colab: Runtime → Secrets, or os.environ['OPENAI_API_KEY']='...')
assert os.getenv('OPENAI_API_KEY'), 'Please set OPENAI_API_KEY before running this cell.'

system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
Hard rules:
1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
Only output in the requested format.
"""
evidence = build_evidence_block("high_risk", top_k=8, alpha=0.5)
user_text = build_user_prompt("high_risk", evidence)

resp = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role":"system", "content": system_text},
        {"role":"user", "content": user_text},
    ],
)

print(resp.output_text)


### Cluster Overview
- The high-risk cluster includes individuals with an average age of 67.4 years, highlighting an older demographic with significant health concerns [C]. 
- Blood pressure levels in this cluster are notably elevated, with a mean of 157.2/102.1 mmHg, indicating that all subjects qualify as hypertensive [C].
- All members of the cluster suffer from obesity, as indicated by a body mass index (BMI) with a mean of 34.1 [C].

### Key Indicators (BP / BMI / Cholesterol / Glucose / Kidney)
- Blood Pressure: Mean 157.2/102.1 mmHg; 100% have BP >= 130/80 and 100% have BP >= 140/90 [C].
- BMI: Mean 34.1; 100% have BMI >= 25 and 100% have BMI >= 30 [C].
- Cholesterol: Mean 270.6; 100% have cholesterol levels >= 200 and 100% have levels >= 240 [C].
- Glucose: Mean 158.2; 100% have glucose levels >= 100 and 100% have levels >= 126 [C].
- Kidney: insufficient evidence

### Main Risk Factors
- Many patients in this cluster exhibit obesity, hypertension, high cholesterol levels, and 

In [ ]:
Evidence_block_high = evidence
print(Evidence_block_high)

EVIDENCE (You MUST cite these IDs; do not invent new ones)

[C] Cluster profile (high_risk):
CLUSTER PROFILE — high_risk (n=52) Age: mean 67.4, range [51.0, 92.0] Blood pressure: mean 157.2/102.1 mmHg; %>=130/80: 100.0%; %>=140/90: 100.0% BMI: mean 34.1; %>=25: 100.0%; %>=30: 100.0% Glucose: mean 158.2; %>=100: 100.0%; %>=126: 100.0% Cholesterol: mean 270.6; %>=200: 100.0%; %>=240: 100.0%

[S1] Patient snippet:
Patient, aged 59.0, has obesity, hypertension, high cholesterol, diabetes risk.

[S2] Patient snippet:
Patient, aged 78.0, has obesity, hypertension, high cholesterol, diabetes risk.

[S3] Patient snippet:
Patient, aged 54.0, has obesity, hypertension, high cholesterol, diabetes risk.

[S4] Patient snippet:
Patient, aged 52.0, has obesity, hypertension, high cholesterol, diabetes risk.

[S5] Patient snippet:
Patient, aged 69.0, has obesity, hypertension, high cholesterol, diabetes risk.

[S6] Patient snippet:
Patient, aged 69.0, has obesity, hypertension, high cholesterol, diabe

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

# NOTE: Set your API key in the environment (Colab: Runtime → Secrets, or os.environ['OPENAI_API_KEY']='...')
assert os.getenv('OPENAI_API_KEY'), 'Please set OPENAI_API_KEY before running this cell.'

system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
Hard rules:
1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
Only output in the requested format.
"""
evidence = build_evidence_block("low_risk", top_k=8, alpha=0.5)
user_text = build_user_prompt("low_risk", evidence)

resp = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role":"system", "content": system_text},
        {"role":"user", "content": user_text},
    ],
)

print(resp.output_text)


### Cluster Overview
- The low_risk cluster consists of individuals with a mean age of 35.6 years and a range from 20.0 to 49.0 years [C]. 
- Blood pressure levels are often within a normal range, with a mean of 106.6/69.2 mmHg and only 5.7% showing elevated blood pressure [C]. 
- Many individuals in this cluster exhibit normal values for body mass index (BMI), cholesterol, and glucose [C].

### Key Indicators (BP / BMI / Cholesterol / Glucose / Kidney)
- Blood Pressure: Mean 106.6/69.2 mmHg; 5.7% ≥130/80 mmHg.
- BMI: Mean 21.5; 0.0% ≥25.
- Cholesterol: Mean 175.0; 0.0% ≥200.
- Glucose: Mean 85.6; 0.0% ≥100.
- Kidney: insufficient evidence.

### Main Risk Factors
- Many individuals in the low_risk cluster show no significant indicators of hypertension or obesity [C]. 
- On average, there are low percentages of elevated cholesterol and glucose levels within this group [C]. 
- A subset of patients have consistently reported normal metabolic profiles, suggesting a trend towards overall me

In [ ]:
Evidence_block_low = evidence
print(Evidence_block_low)

EVIDENCE (You MUST cite these IDs; do not invent new ones)

[C] Cluster profile (low_risk):
CLUSTER PROFILE — low_risk (n=35) Age: mean 35.6, range [20.0, 49.0] Blood pressure: mean 106.6/69.2 mmHg; %>=130/80: 5.7%; %>=140/90: 0.0% BMI: mean 21.5; %>=25: 0.0%; %>=30: 0.0% Glucose: mean 85.6; %>=100: 0.0%; %>=126: 0.0% Cholesterol: mean 175.0; %>=200: 0.0%; %>=240: 0.0%

[S1] Patient snippet:
Patient, aged 23.0, has a normal BMI, normal blood pressure, normal cholesterol, normal glucose levels.

[S2] Patient snippet:
Patient, aged 21.0, has a normal BMI, normal blood pressure, normal cholesterol, normal glucose levels.

[S3] Patient snippet:
Patient, aged 21.0, has a normal BMI, normal blood pressure, normal cholesterol, normal glucose levels.

[S4] Patient snippet:
Patient, aged 34.0, has a normal BMI, normal blood pressure, normal cholesterol, normal glucose levels.

[S5] Patient snippet:
Patient, aged 26.0, has a normal BMI, normal blood pressure, normal cholesterol, normal glucose l

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

# NOTE: Set your API key in the environment (Colab: Runtime → Secrets, or os.environ['OPENAI_API_KEY']='...')
assert os.getenv('OPENAI_API_KEY'), 'Please set OPENAI_API_KEY before running this cell.'

system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
Hard rules:
1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
Only output in the requested format.
"""
evidence = build_evidence_block("average_risk", top_k=8, alpha=0.5)
user_text = build_user_prompt("average_risk", evidence)

resp = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {"role":"system", "content": system_text},
        {"role":"user", "content": user_text},
    ],
)

print(resp.output_text)


### Cluster Overview
- The average risk cluster consists of 62 individuals with a mean age of approximately 57.8 years, indicating a mature demographic [C]. 
- Within this cluster, blood pressure readings show that all individuals have a value of at least 130/80 mmHg, with a notable subset exhibiting readings at or above 140/90 mmHg [C].

### Key Indicators (BP / BMI / Cholesterol / Glucose / Kidney)
- Blood Pressure: Mean of 134.6/85.7 mmHg; 100% ≥ 130/80 mmHg; 22.6% ≥ 140/90 mmHg [C].
- BMI: Mean of 28.4; 100% ≥ 25; 14.5% ≥ 30 [C].
- Cholesterol: Mean of 228.6; 100% ≥ 200; 14.5% ≥ 240 [C].
- Glucose: Mean of 118.6; 100% ≥ 100; 14.5% ≥ 126 [C].
- Kidney: insufficient evidence.

### Main Risk Factors
- Many individuals in this cluster display indicators of borderline hypertension, as observed in multiple patient snippets [S1-S8].
- There is also a tendency toward borderline obesity, with a BMI often reported around 28.4 [C].
- High cholesterol levels are prevalent, with all cluster mem

In [ ]:
Evidence_block_average = evidence
print(Evidence_block_average)

EVIDENCE (You MUST cite these IDs; do not invent new ones)

[C] Cluster profile (average_risk):
CLUSTER PROFILE — average_risk (n=62) Age: mean 57.8, range [35.0, 93.0] Blood pressure: mean 134.6/85.7 mmHg; %>=130/80: 100.0%; %>=140/90: 22.6% BMI: mean 28.4; %>=25: 100.0%; %>=30: 14.5% Glucose: mean 118.6; %>=100: 100.0%; %>=126: 14.5% Cholesterol: mean 228.6; %>=200: 100.0%; %>=240: 14.5%

[S1] Patient snippet:
Patient, aged 91.0, has a borderline obesity BMI, borderline hypertension, borderline high cholesterol, and borderline diabetes.

[S2] Patient snippet:
Patient, aged 59.0, has a borderline obesity BMI, borderline hypertension, borderline high cholesterol, and borderline diabetes.

[S3] Patient snippet:
Patient, aged 76.0, has a borderline obesity BMI, borderline hypertension, borderline high cholesterol, and borderline diabetes.

[S4] Patient snippet:
Patient, aged 52.0, has a borderline obesity BMI, borderline hypertension, borderline high cholesterol, and borderline diabetes.

Comaring the S1 to above Hybrid RAG results

In [ ]:
#!/usr/bin/env python3
import json, re
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROFILE_TABLE = "cluster_profile_table.csv"

profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese", "pct_Glucose_prediabetes", "pct_Glucose_diabetes", "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "pre-hypertens", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b", r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bhas (?:a|an|the)\b.*\b(disease|disorder|condition)\b", r"\bchronic kidney disease\b", r"\bckd\b", r"\bmyocardial infarction\b", r"\bstroke\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdrug\b", r"\bdosage\b", r"\bmetformin\b", r"\bstatin\b", r"\binsulin\b", r"\bantihypertensive\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def tfidf_cosine(a: str, b: str):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform([a, b])
    return float(cosine_similarity(X[0], X[1])[0,0])

def extract_claims(summary_text: str, min_chars=20, max_claims=15):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    if len(claims) > max_claims:
        claims = claims[:max_claims]
    return claims

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min())/(a.max()-a.min()+eps)

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E = X[:len(e_texts)]
    C = X[len(e_texts):]
    sims = cosine_similarity(C, E)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i,j])
        results.append({
            "claim": claim,
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds)), hits

def count_patterns(text, patterns):
    t = text.lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, t))
    return total

def evaluate_one(item, n_claims=12):
    cluster = item["cluster"]
    rag = item["rag_summary"]
    ref = item["reference_s1"]
    ev  = item["evidence_block"]
    sim = tfidf_cosine(rag, ref)

    evidence_items = parse_evidence_block(ev)
    claims = extract_claims(rag, max_claims=n_claims)
    cr = support_check_claims(claims, evidence_items)
    supported = sum(1 for r in cr if r["supported"])
    total = len(cr)
    supported_rate = (supported/total*100) if total else 0.0
    unsupported_rate = 100-supported_rate if total else 0.0

    must = pick_must_mentions(cluster)
    comp, comp_hits = completeness_score(rag, must)

    abs_ct = count_patterns(rag, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag, MEDICATION_PATTERNS)

    return {
        "cluster": cluster,
        "similarity_tfidf": sim,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "n_claims_checked": total,
        "completeness_%": comp*100,
        "must_mentions": ";".join(must),
        "must_mentions_hit": comp_hits,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct
    }

def main():
    with open("step7_eval_input.json","r",encoding="utf-8") as f:
        items = json.load(f)
    rows = [evaluate_one(it) for it in items]
    out = pd.DataFrame(rows)
    out.to_csv("step7_evaluation_results.csv", index=False)
    print(out)

if __name__ == "__main__":
    main()


        cluster  similarity_tfidf  supported_claim_rate_%  \
0      low_risk          0.206744               50.000000   
1  average_risk          0.070449               33.333333   
2     high_risk          0.120425               33.333333   

   unsupported_claim_rate_%  n_claims_checked  completeness_%  \
0                 50.000000                12           100.0   
1                 66.666667                12           100.0   
2                 66.666667                12           100.0   

                                  must_mentions  must_mentions_hit  \
0  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   
1  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   
2  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   

   absolute_claim_flags  diagnosis_like_flags  medication_flags  
0                     0                     0                 0  
1                     2                     0                 0  
2              

Comaring the TSNE_PSO to above Hybrid RAG results

In [ ]:
#!/usr/bin/env python3
import json, re
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROFILE_TABLE = "cluster_profile_table.csv"

profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese", "pct_Glucose_prediabetes", "pct_Glucose_diabetes", "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "pre-hypertens", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b", r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bhas (?:a|an|the)\b.*\b(disease|disorder|condition)\b", r"\bchronic kidney disease\b", r"\bckd\b", r"\bmyocardial infarction\b", r"\bstroke\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdrug\b", r"\bdosage\b", r"\bmetformin\b", r"\bstatin\b", r"\binsulin\b", r"\bantihypertensive\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def tfidf_cosine(a: str, b: str):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform([a, b])
    return float(cosine_similarity(X[0], X[1])[0,0])

def extract_claims(summary_text: str, min_chars=20, max_claims=15):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    if len(claims) > max_claims:
        claims = claims[:max_claims]
    return claims

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min())/(a.max()-a.min()+eps)

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E = X[:len(e_texts)]
    C = X[len(e_texts):]
    sims = cosine_similarity(C, E)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i,j])
        results.append({
            "claim": claim,
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds)), hits

def count_patterns(text, patterns):
    t = text.lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, t))
    return total

def evaluate_one(item, n_claims=12):
    cluster = item["cluster"]
    rag = item["rag_summary"]
    ref = item["reference_s1"]
    ev  = item["evidence_block"]
    sim = tfidf_cosine(rag, ref)

    evidence_items = parse_evidence_block(ev)
    claims = extract_claims(rag, max_claims=n_claims)
    cr = support_check_claims(claims, evidence_items)
    supported = sum(1 for r in cr if r["supported"])
    total = len(cr)
    supported_rate = (supported/total*100) if total else 0.0
    unsupported_rate = 100-supported_rate if total else 0.0

    must = pick_must_mentions(cluster)
    comp, comp_hits = completeness_score(rag, must)

    abs_ct = count_patterns(rag, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag, MEDICATION_PATTERNS)

    return {
        "cluster": cluster,
        "similarity_tfidf": sim,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "n_claims_checked": total,
        "completeness_%": comp*100,
        "must_mentions": ";".join(must),
        "must_mentions_hit": comp_hits,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct
    }

def main():
    with open("step7_eval_input.json","r",encoding="utf-8") as f:
        items = json.load(f)
    rows = [evaluate_one(it) for it in items]
    out = pd.DataFrame(rows)
    out.to_csv("step7_evaluation_results.csv", index=False)
    print(out)

if __name__ == "__main__":
    main()


        cluster  similarity_tfidf  supported_claim_rate_%  \
0      low_risk          0.230017               50.000000   
1  average_risk          0.076159               33.333333   
2     high_risk          0.128792               33.333333   

   unsupported_claim_rate_%  n_claims_checked  completeness_%  \
0                 50.000000                12           100.0   
1                 66.666667                12           100.0   
2                 66.666667                12           100.0   

                                  must_mentions  must_mentions_hit  \
0  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   
1  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   
2  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   

   absolute_claim_flags  diagnosis_like_flags  medication_flags  
0                     0                     0                 0  
1                     2                     0                 0  
2              

For S1 base line and S4 base line

In [ ]:
#!/usr/bin/env python3
import json, re
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROFILE_TABLE = "cluster_profile_table.csv"

profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese", "pct_Glucose_prediabetes", "pct_Glucose_diabetes", "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "pre-hypertens", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b", r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bhas (?:a|an|the)\b.*\b(disease|disorder|condition)\b", r"\bchronic kidney disease\b", r"\bckd\b", r"\bmyocardial infarction\b", r"\bstroke\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdrug\b", r"\bdosage\b", r"\bmetformin\b", r"\bstatin\b", r"\binsulin\b", r"\bantihypertensive\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def tfidf_cosine(a: str, b: str):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform([a, b])
    return float(cosine_similarity(X[0], X[1])[0,0])

def extract_claims(summary_text: str, min_chars=20, max_claims=15):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    if len(claims) > max_claims:
        claims = claims[:max_claims]
    return claims

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min())/(a.max()-a.min()+eps)

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E = X[:len(e_texts)]
    C = X[len(e_texts):]
    sims = cosine_similarity(C, E)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i,j])
        results.append({
            "claim": claim,
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds)), hits

def count_patterns(text, patterns):
    t = text.lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, t))
    return total

def evaluate_one(item, n_claims=12):
    cluster = item["cluster"]
    rag = item["rag_summary"]
    ref = item["reference_s1"]
    ev  = item["evidence_block"]
    sim = tfidf_cosine(rag, ref)

    evidence_items = parse_evidence_block(ev)
    claims = extract_claims(rag, max_claims=n_claims)
    cr = support_check_claims(claims, evidence_items)
    supported = sum(1 for r in cr if r["supported"])
    total = len(cr)
    supported_rate = (supported/total*100) if total else 0.0
    unsupported_rate = 100-supported_rate if total else 0.0

    must = pick_must_mentions(cluster)
    comp, comp_hits = completeness_score(rag, must)

    abs_ct = count_patterns(rag, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag, MEDICATION_PATTERNS)

    return {
        "cluster": cluster,
        "similarity_tfidf": sim,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "n_claims_checked": total,
        "completeness_%": comp*100,
        "must_mentions": ";".join(must),
        "must_mentions_hit": comp_hits,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct
    }

def main():
    with open("step7_eval_input.json","r",encoding="utf-8") as f:
        items = json.load(f)
    rows = [evaluate_one(it) for it in items]
    out = pd.DataFrame(rows)
    out.to_csv("step7_evaluation_results.csv", index=False)
    print(out)

if __name__ == "__main__":
    main()


        cluster  similarity_tfidf  supported_claim_rate_%  \
0      low_risk          0.327408                    25.0   
1  average_risk          0.323532                     0.0   
2     high_risk          0.215736                     0.0   

   unsupported_claim_rate_%  n_claims_checked  completeness_%  \
0                      75.0                 4           100.0   
1                     100.0                 4           100.0   
2                     100.0                 4            50.0   

                                  must_mentions  must_mentions_hit  \
0  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   
1  pct_BP_elevated_or_higher;pct_BMI_overweight                  2   
2  pct_BP_elevated_or_higher;pct_BMI_overweight                  1   

   absolute_claim_flags  diagnosis_like_flags  medication_flags  
0                     0                     0                 0  
1                     0                     0                 0  
2              

now calculate the delta


In [ ]:
import pandas as pd

rag_s1 = pd.read_csv("step7_evaluation_results_S1vsRAG.csv").set_index("cluster")
base_s1 = pd.read_csv("step7_evaluation_results_S1baseline.csv").set_index("cluster")

rag_s4 = pd.read_csv("step7_evaluation_results_S4vsRAG.csv").set_index("cluster")
base_s4 = pd.read_csv("step7_evaluation_results_S4baseline.csv").set_index("cluster")

def delta(rag, base, label):
    out = pd.DataFrame(index=rag.index)
    out["Comparison"] = label
    out["ΔSupported(%)"] = rag["supported_claim_rate_%"] - base["supported_claim_rate_%"]
    out["ΔUnsupported(%)"] = base["unsupported_claim_rate_%"] - rag["unsupported_claim_rate_%"]
    out["ΔCompleteness(%)"] = rag["completeness_%"] - base["completeness_%"]
    out["ΔAbsoluteFlags"] = base["absolute_claim_flags"] - rag["absolute_claim_flags"]
    return out.reset_index()

print(pd.concat([
    delta(rag_s1, base_s1, "S1 baseline → S1-RAG"),
    delta(rag_s4, base_s4, "S4 baseline → S4-RAG"),
], ignore_index=True))


        cluster            Comparison  ΔSupported(%)  ΔUnsupported(%)  \
0      low_risk  S1 baseline → S1-RAG      16.666667        16.666667   
1  average_risk  S1 baseline → S1-RAG      33.333333        33.333333   
2     high_risk  S1 baseline → S1-RAG      33.333333        33.333333   
3      low_risk  S4 baseline → S4-RAG      25.000000        25.000000   
4  average_risk  S4 baseline → S4-RAG      33.333333        33.333333   
5     high_risk  S4 baseline → S4-RAG      33.333333        33.333333   

   ΔCompleteness(%)  ΔAbsoluteFlags  
0               0.0               0  
1               0.0              -2  
2              50.0              -2  
3               0.0               0  
4               0.0              -2  
5              50.0              -2  
